# Uçuş Gecikmesi Tahmini — 1. Hava durumu verisi ve havalimanı eşleştirmesi
Bu notebook üç tabloyu birbirine bağlar:

| Tablo | Kaynak | Ne işe yarıyor |
|---|---|---|
| Uçuşlar (2025) | Kaggle — BTS | Hangi havalimanları var? (IATA kodu: JFK, ORD…) |
| Havalimanları | OurAirports (açık havalimanı listesi) | IATA kodu → enlem / boylam |
| Hava istasyonları + günlük hava durumu | AWS Open Data — NOAA GHCN-Daily | Koordinat → en yakın istasyon → günlük sıcaklık, yağış, kar, rüzgar |

**Ön koşul:** `00_ucus_veri_kesfi` çalışmış olmalı (uçuş dosyaları Drive'da `UcusGecikme/raw/flights_2025` klasöründe).

**Çıktılar** (Drive `UcusGecikme/processed`): `airports.parquet` (havalimanı → istasyon eşleşmesi) ve `weather_daily_2025.parquet` (havalimanı × gün hava durumu).

Çalışma zamanı → Tümünü çalıştır. NOAA dosyası büyük; ilk çalıştırmada indirme ve okuma 5–10 dakika sürebilir. Bitince .ipynb olarak indirip gönder.

## 0. Kurulum

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'duckdb'], check=False)

from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import requests

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/UcusGecikme')
except ImportError:
    BASE = Path('.')

FLIGHTS = BASE / 'raw' / 'flights_2025'
WEATHER_RAW = BASE / 'raw' / 'weather'
PROC = BASE / 'processed'
for p in (WEATHER_RAW, PROC):
    p.mkdir(parents=True, exist_ok=True)
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

con = duckdb.connect()
flights_sql = f"read_csv_auto('{FLIGHTS}/*.csv', union_by_name=true, sample_size=-1)"
print('Uçuş dosyası sayısı:', len(list(FLIGHTS.glob('*.csv'))))


def download(urls, dest):
    """Dosya Drive'da yoksa, listedeki adresleri sırayla dener ve ilk çalışanı indirir."""
    if dest.exists() and dest.stat().st_size > 0:
        print(f'Zaten var: {dest.name} ({dest.stat().st_size / 1e6:,.1f} MB)')
        return dest
    for url in urls:
        try:
            r = requests.get(url, stream=True, timeout=60)
            if r.status_code != 200:
                print(f'  {r.status_code} → {url}')
                continue
            tmp = dest.with_suffix(dest.suffix + '.part')
            with open(tmp, 'wb') as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
            tmp.rename(dest)
            print(f'İndirildi: {url} → {dest.name} ({dest.stat().st_size / 1e6:,.1f} MB)')
            return dest
        except requests.RequestException as e:
            print(f'  hata → {url}: {e}')
    raise RuntimeError(f'{dest.name} indirilemedi; adresleri kontrol et.')

Mounted at /content/drive
Uçuş dosyası sayısı: 12


## 1. Uçuş verisindeki havalimanları

In [2]:
ap_flights = con.sql(f'''
    SELECT code, any_value(city) AS city, any_value(state) AS state, sum(n) AS ucus
    FROM (SELECT Origin AS code, OriginCityName AS city, OriginState AS state, count(*) AS n FROM {flights_sql} GROUP BY ALL
          UNION ALL
          SELECT Dest, DestCityName, DestState, count(*) FROM {flights_sql} GROUP BY ALL)
    GROUP BY code ORDER BY ucus DESC''').df()
print(f'Uçuş verisinde {len(ap_flights)} havalimanı var.')
print(ap_flights.head(5).to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Uçuş verisinde 352 havalimanı var.
code                  city state     ucus
 ORD           Chicago, IL    IL 654039.0
 DEN            Denver, CO    CO 635361.0
 DFW Dallas/Fort Worth, TX    TX 631706.0
 ATL           Atlanta, GA    GA 628872.0
 PHX           Phoenix, AZ    AZ 393513.0


## 2. Havalimanı koordinatları (OurAirports)

In [3]:
ap_file = download(['https://raw.githubusercontent.com/davidmegginson/ourairports-data/main/airports.csv',
                    'https://davidmegginson.github.io/ourairports-data/airports.csv'],
                   BASE / 'raw' / 'ourairports_airports.csv')
oa = pd.read_csv(ap_file, usecols=['ident', 'type', 'name', 'latitude_deg', 'longitude_deg', 'iso_country', 'iata_code'])
oa = oa.dropna(subset=['iata_code'])
# Aynı IATA kodu birden fazla satırda olabilir (kapanmış havalimanı vb.) → büyük havalimanını tercih et
type_rank = {'large_airport': 0, 'medium_airport': 1, 'small_airport': 2}
oa['rank'] = oa['type'].map(type_rank).fillna(9)
print('OurAirports\'ta birden fazla satırı olan IATA kodu:', oa['iata_code'].duplicated().sum())
oa = oa.sort_values('rank').drop_duplicates('iata_code')

airports = ap_flights.merge(oa.rename(columns={'iata_code': 'code', 'name': 'ap_name', 'latitude_deg': 'lat',
                                               'longitude_deg': 'lon', 'ident': 'icao'}),
                            on='code', how='left')
missing = airports[airports['lat'].isna()]
print(f'Koordinatı bulunan: {airports["lat"].notna().sum()} / {len(airports)}')
if len(missing):
    print('Koordinatı BULUNAMAYAN havalimanları:')
    print(missing[['code', 'city', 'state', 'ucus']].to_string(index=False))

İndirildi: https://raw.githubusercontent.com/davidmegginson/ourairports-data/main/airports.csv → ourairports_airports.csv (12.7 MB)
OurAirports'ta birden fazla satırı olan IATA kodu: 0
Koordinatı bulunan: 351 / 352
Koordinatı BULUNAMAYAN havalimanları:
code                           city state    ucus
 PBI West Palm Beach/Palm Beach, FL    FL 61573.0


## 3. NOAA hava istasyonları
`ghcnd-stations.txt` sabit genişlikli bir metin dosyası: her satırda istasyon kodu, enlem, boylam, yükseklik, eyalet ve adı var.
**USW** ile başlayan istasyonlar ABD Ulusal Hava Servisi'nin ana istasyonları; çoğu havalimanlarında.

In [4]:
st_file = download(['https://noaa-ghcn-pds.s3.amazonaws.com/ghcnd-stations.txt'], WEATHER_RAW / 'ghcnd-stations.txt')
stations = pd.read_fwf(st_file, colspecs=[(0, 11), (12, 20), (21, 30), (31, 37), (38, 40), (41, 71)],
                       names=['station', 'st_lat', 'st_lon', 'elev', 'st_state', 'st_name'], dtype={'station': str})
# ABD ve bağlı bölgeler: US, Porto Riko (RQ), Virjin Adaları (VQ), Guam (GQ), Kuzey Mariana (CQ), Amerikan Samoası (AQ)
stations = stations[stations['station'].str[:2].isin(['US', 'RQ', 'VQ', 'GQ', 'CQ', 'AQ'])]
print(f'ABD ve bağlı bölgelerdeki istasyon: {len(stations):,}   (USW ana istasyon: {stations["station"].str.startswith("USW").sum():,})')

İndirildi: https://noaa-ghcn-pds.s3.amazonaws.com/ghcnd-stations.txt → ghcnd-stations.txt (11.4 MB)
ABD ve bağlı bölgelerdeki istasyon: 78,999   (USW ana istasyon: 1,909)


## 4. NOAA 2025 günlük hava durumu
Dosya tüm dünyanın 2025 verisini içeriyor (on milyonlarca satır). DuckDB ile sadece ABD istasyonlarını ve işimize yarayan ölçümleri alıyoruz.

| Kod | Anlamı | Birim (ham) |
|---|---|---|
| TMAX / TMIN | En yüksek / en düşük sıcaklık | 0,1 °C |
| PRCP | Yağış | 0,1 mm |
| SNOW / SNWD | Yağan kar / yerdeki kar kalınlığı | mm |
| AWND | Ortalama rüzgar hızı | 0,1 m/s |
| WSF2 | En hızlı 2 dakikalık rüzgar | 0,1 m/s |
| WT01 / WT03 / WT05 | Sis / gök gürültülü fırtına / dolu (olduysa 1) | — |

In [5]:
wx_file = download(['https://noaa-ghcn-pds.s3.amazonaws.com/csv.gz/by_year/2025.csv.gz',
                    'https://noaa-ghcn-pds.s3.amazonaws.com/csv.gz/2025.csv.gz',
                    'https://noaa-ghcn-pds.s3.amazonaws.com/csv/by_year/2025.csv',
                    'https://noaa-ghcn-pds.s3.amazonaws.com/csv/2025.csv'],
                   WEATHER_RAW / 'ghcnd_2025.csv.gz')

ELEMENTS = ['TMAX', 'TMIN', 'PRCP', 'SNOW', 'SNWD', 'AWND', 'WSF2', 'WT01', 'WT03', 'WT05']
raw_us = PROC / 'ghcnd_2025_us_long.parquet'
if not raw_us.exists():
    con.sql(f'''
        COPY (
          SELECT station, strptime(CAST(date AS VARCHAR), '%Y%m%d')::DATE AS date, element, value, q_flag
          FROM read_csv('{wx_file}', header=false, all_varchar=false,
                        columns={{'station':'VARCHAR','date':'INTEGER','element':'VARCHAR','value':'DOUBLE',
                                  'm_flag':'VARCHAR','q_flag':'VARCHAR','s_flag':'VARCHAR','obs_time':'VARCHAR'}})
          WHERE substr(station, 1, 2) IN ('US','RQ','VQ','GQ','CQ','AQ')
            AND element IN ({', '.join(repr(e) for e in ELEMENTS)})
        ) TO '{raw_us}' (FORMAT parquet)''')
long = pd.read_parquet(raw_us)
print(f'ABD 2025 ölçüm satırı: {len(long):,}')
print(f'Kalite kontrolünden geçemeyen (q_flag dolu) ölçüm: {long["q_flag"].notna().sum():,} (%{long["q_flag"].notna().mean() * 100:.2f}) → bunları atıyoruz')
long = long[long['q_flag'].isna()]
print(f'Tarih aralığı: {long["date"].min()} → {long["date"].max()}')

İndirildi: https://noaa-ghcn-pds.s3.amazonaws.com/csv.gz/by_year/2025.csv.gz → ghcnd_2025.csv.gz (167.3 MB)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ABD 2025 ölçüm satırı: 22,406,783
Kalite kontrolünden geçemeyen (q_flag dolu) ölçüm: 16,850 (%0.08) → bunları atıyoruz
Tarih aralığı: 2025-01-01 → 2025-12-31


## 5. Her havalimanına en yakın istasyonu seçme
Kural: 2025'te **sıcaklık ve yağışı en az 300 gün** ölçmüş istasyonlar arasından en yakını.
25 km içinde bir **USW** (ana istasyon) varsa onu tercih ediyoruz; yoksa en yakın uygun istasyon.

In [6]:
cov = (long[long['element'].isin(['TMAX', 'PRCP'])]
       .groupby(['station', 'element'])['date'].nunique().unstack(fill_value=0))
good = cov[(cov.get('TMAX', 0) >= 300) & (cov.get('PRCP', 0) >= 300)].index
cand = stations[stations['station'].isin(good)].reset_index(drop=True)
print(f'Yeterli ölçümü olan aday istasyon: {len(cand):,}')


def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))


is_usw = cand['station'].str.startswith('USW').values
rows = []
for r in airports.dropna(subset=['lat']).itertuples():
    d = haversine_km(r.lat, r.lon, cand['st_lat'].values, cand['st_lon'].values)
    near_usw = np.where(is_usw & (d <= 25))[0]
    i = near_usw[np.argmin(d[near_usw])] if len(near_usw) else int(np.argmin(d))
    rows.append({'code': r.code, 'station': cand.at[i, 'station'], 'st_name': cand.at[i, 'st_name'],
                 'distance_km': round(float(d[i]), 1)})
match = pd.DataFrame(rows)
airports = airports.merge(match, on='code', how='left')

print('İstasyona uzaklık (km) özeti:')
print(airports['distance_km'].describe(percentiles=[.5, .9, .95]).round(1).to_string())
print(f'\nUSW (ana istasyon) ile eşleşen havalimanı: {airports["station"].str.startswith("USW", na=False).sum()} / {airports["station"].notna().sum()}')
print('\nEn yoğun 10 havalimanının istasyonu:')
print(airports.head(10)[['code', 'city', 'station', 'st_name', 'distance_km']].to_string(index=False))
print('\nİstasyonu en UZAK 10 havalimanı (dikkat!):')
print(airports.sort_values('distance_km', ascending=False).head(10)[['code', 'city', 'ucus', 'st_name', 'distance_km']].to_string(index=False))

Yeterli ölçümü olan aday istasyon: 5,191
İstasyona uzaklık (km) özeti:
count    351.0
mean       4.9
std       38.9
min        0.0
50%        0.8
90%        3.9
95%       18.2
max      717.2

USW (ana istasyon) ile eşleşen havalimanı: 317 / 351

En yoğun 10 havalimanının istasyonu:
code                  city     station                        st_name  distance_km
 ORD           Chicago, IL USW00094846          CHICAGO OHARE INTL AP          3.0
 DEN            Denver, CO USW00003017                 DENVER INTL AP          2.1
 DFW Dallas/Fort Worth, TX USW00003927               DAL-FTW WSCMO AP          1.5
 ATL           Atlanta, GA USW00013874 ATLANTA HARTSFIELD-JACKSON INT          1.5
 PHX           Phoenix, AZ USW00023183                     PHOENIX AP          0.9
 CLT         Charlotte, NC USW00013881           CHARLOTTE DOUGLAS AP          1.4
 LAX       Los Angeles, CA USW00023174            LOS ANGELES INTL AP          2.0
 LAS         Las Vegas, NV USW00023169               

## 6. Günlük hava durumu tablosu: havalimanı × gün
Uzun formattaki ölçümleri (her satır bir ölçüm) geniş formata çeviriyoruz (her satır bir gün) ve birimleri düzeltiyoruz.

In [7]:
sel = long[long['station'].isin(airports['station'].dropna().unique())]
wide = sel.pivot_table(index=['station', 'date'], columns='element', values='value', aggfunc='first').reset_index()
for e in ELEMENTS:
    if e not in wide:
        wide[e] = np.nan
wx = pd.DataFrame({
    'station': wide['station'], 'date': pd.to_datetime(wide['date']),
    'tmax_c': wide['TMAX'] / 10, 'tmin_c': wide['TMIN'] / 10,
    'prcp_mm': wide['PRCP'] / 10, 'snow_mm': wide['SNOW'], 'snwd_mm': wide['SNWD'],
    'awnd_ms': wide['AWND'] / 10, 'wsf2_ms': wide['WSF2'] / 10,
    # Hava olayı kodları sadece olay olduğunda yazılıyor → yoksa 0
    'fog': wide['WT01'].notna().astype(int), 'thunder': wide['WT03'].notna().astype(int), 'hail': wide['WT05'].notna().astype(int),
})
weather = airports[['code', 'station']].dropna().merge(wx, on='station', how='left')
weather = weather.rename(columns={'code': 'airport'})

print(f'Hava durumu tablosu: {len(weather):,} satır ({weather["airport"].nunique()} havalimanı × gün)')
print('\nEksik oranı (%):')
print((weather.isna().mean() * 100).round(1).to_string())
print('\nÖrnek: Chicago O\'Hare (ORD), ilk 5 gün')
print(weather[weather['airport'] == 'ORD'].sort_values('date').head().to_string(index=False))

Hava durumu tablosu: 127,728 satır (351 havalimanı × gün)

Eksik oranı (%):
airport     0.0
station     0.0
date        0.0
tmax_c      0.2
tmin_c      0.3
prcp_mm     0.6
snow_mm    33.5
snwd_mm    37.0
awnd_ms    11.8
wsf2_ms    11.7
fog         0.0
thunder     0.0
hail        0.0

Örnek: Chicago O'Hare (ORD), ilk 5 gün
airport     station       date  tmax_c  tmin_c  prcp_mm  snow_mm  snwd_mm  awnd_ms  wsf2_ms  fog  thunder  hail
    ORD USW00094846 2025-01-01     1.1    -4.3      0.0      0.0      0.0      5.9      9.8    1        0     0
    ORD USW00094846 2025-01-02     0.0    -4.3      0.3      3.0      0.0      4.1      7.2    1        0     0
    ORD USW00094846 2025-01-03    -3.8    -7.7      0.0      5.0      0.0      5.3      8.9    1        0     0
    ORD USW00094846 2025-01-04    -3.8   -11.0      0.0      0.0      0.0      4.2      6.7    0        0     0
    ORD USW00094846 2025-01-05    -5.5    -9.3      0.0      0.0      0.0      4.1      8.1    1        0     0


## 7. Birleştirme testi: uçuşların yüzde kaçı hava durumu bilgisi alabiliyor?

In [8]:
con.register('weather', weather)
res = con.sql(f'''
    SELECT count(*) AS ucus,
           avg(CASE WHEN w.airport IS NOT NULL THEN 1 ELSE 0 END) * 100 AS kalkis_eslesen_yuzde,
           avg(CASE WHEN w.tmax_c IS NOT NULL THEN 1 ELSE 0 END) * 100 AS sicaklik_dolu_yuzde,
           avg(CASE WHEN w.prcp_mm IS NOT NULL THEN 1 ELSE 0 END) * 100 AS yagis_dolu_yuzde,
           avg(CASE WHEN w.awnd_ms IS NOT NULL THEN 1 ELSE 0 END) * 100 AS ruzgar_dolu_yuzde
    FROM {flights_sql} f
    LEFT JOIN weather w ON f.Origin = w.airport AND CAST(f.FlightDate AS DATE) = CAST(w.date AS DATE)''').df()
print(res.round(1).to_string(index=False))

print('\nİlk bakış: kalkış havalimanında gök gürültülü fırtına olan günlerde gecikme oranı')
print(con.sql(f'''
    SELECT w.thunder AS firtina, count(*) AS ucus, round(avg(f.ArrDel15) * 100, 1) AS gecikme_yuzde
    FROM {flights_sql} f JOIN weather w ON f.Origin = w.airport AND CAST(f.FlightDate AS DATE) = CAST(w.date AS DATE)
    WHERE f.ArrDel15 IS NOT NULL GROUP BY 1 ORDER BY 1''').df().to_string(index=False))
print('\nKalkış günü kar yağışına göre gecikme oranı')
print(con.sql(f'''
    SELECT CASE WHEN w.snow_mm IS NULL THEN 'bilinmiyor' WHEN w.snow_mm = 0 THEN 'kar yok'
                WHEN w.snow_mm < 50 THEN '0-5 cm' ELSE '5 cm üstü' END AS kar,
           count(*) AS ucus, round(avg(f.ArrDel15) * 100, 1) AS gecikme_yuzde
    FROM {flights_sql} f JOIN weather w ON f.Origin = w.airport AND CAST(f.FlightDate AS DATE) = CAST(w.date AS DATE)
    WHERE f.ArrDel15 IS NOT NULL GROUP BY 1 ORDER BY 2 DESC''').df().to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   ucus  kalkis_eslesen_yuzde  sicaklik_dolu_yuzde  yagis_dolu_yuzde  ruzgar_dolu_yuzde
7001619                  99.5                 99.4              99.4               98.1

İlk bakış: kalkış havalimanında gök gürültülü fırtına olan günlerde gecikme oranı


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 firtina    ucus  gecikme_yuzde
       0 6011615           20.7
       1  833226           34.2

Kalkış günü kar yağışına göre gecikme oranı


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

       kar    ucus  gecikme_yuzde
   kar yok 5540408           22.4
bilinmiyor 1144764           18.7
    0-5 cm  128247           41.6
 5 cm üstü   31422           60.3


## 8. Kaydet

In [9]:
airports.to_parquet(PROC / 'airports.parquet', index=False)
weather.to_parquet(PROC / 'weather_daily_2025.parquet', index=False)
for f in ('airports.parquet', 'weather_daily_2025.parquet'):
    print(f'Kaydedildi: {PROC / f}  ({(PROC / f).stat().st_size / 1e6:.1f} MB)')

Kaydedildi: /content/drive/MyDrive/UcusGecikme/processed/airports.parquet  (0.0 MB)
Kaydedildi: /content/drive/MyDrive/UcusGecikme/processed/weather_daily_2025.parquet  (0.7 MB)


## Bitti
**Dosya → İndir → .ipynb** ile indirip gönder. Sonraki adım: uçuşları temizleyip hava durumuyla birleştiren ana tabloyu hazırlamak.